# CI RAG Pipeline — Advanced Modular RAG with PubMedBERT + FAISS + BM25

In [ ]:
import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))
sys.path.insert(0, str(CT_ROOT / 'src' / 'rag'))

from ct_api import fetch_competing_trials
from rag.chunker import chunk_competing_trials, parse_criteria_via_llm
from rag.embedder import load_pubmedbert, embed_chunks
from rag.index import build_index, load_index
from rag.retriever import retrieve
from rag.retrieval_evaluator import evaluate_retrieval
from rag.ci_reasoner import reason_about_criterion
from rag.recommendation_evaluator import evaluate_recommendations, format_report
from rag.ci_rag_pipeline import run_ci_rag

print(f'CT_ROOT : {CT_ROOT}')

In [ ]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

In [ ]:
# Paste I/E criteria here
ie_criteria = ""


In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — live ClinicalTrials.gov API call
competing_trials_md = fetch_competing_trials(ie_criteria)
print(f'Trials retrieved: {competing_trials_md.count("NCT")} trials')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — LLM call per trial
competitor_chunks = chunk_competing_trials(competing_trials_md)
print(f'Total chunks: {len(competitor_chunks)}')
treatment_chunks = [c for c in competitor_chunks if c.metadata.get("treatment_related", False)]
print(f'Treatment-related chunks: {len(treatment_chunks)}')
trials_represented = len(set(c.source_nct_id for c in competitor_chunks if c.source_nct_id))
print(f'Trials represented: {trials_represented}')

In [ ]:
# Load PubMedBERT and embed all chunks
model = load_pubmedbert()
embeddings = embed_chunks(competitor_chunks, model)
print(f'Embeddings shape: {embeddings.shape}')

# Build hybrid index (FAISS + BM25)
PERSIST_DIR = CT_ROOT / 'data' / 'indexes'
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
hybrid_index = build_index(competitor_chunks, embeddings, 'mariposa2_ci', str(PERSIST_DIR))
print(f'Index built: {hybrid_index.faiss_index.ntotal} vectors indexed')

In [ ]:
# Test retrieval on first treatment-related source criterion
source_chunks = parse_criteria_via_llm(ie_criteria, 'SOURCE', 'Source Trial', {})
treatment_source = [c for c in source_chunks if c.metadata.get('treatment_related', False)]
print(f'Source treatment-related criteria: {len(treatment_source)}')

if treatment_source:
    test_criterion = treatment_source[0]
    print(f'\nTest criterion: {test_criterion.text[:200]}')
    results = retrieve(test_criterion.text, hybrid_index, model, k=5)
    print(f'\nTop 5 retrieved chunks:')
    for r in results:
        print(f'  [{r.rrf_score:.4f}] {r.chunk.source_nct_id} ({r.chunk.criterion_type}): {r.chunk.text[:100]}')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — Claude Sonnet API call
if treatment_source:
    recommendation = reason_about_criterion(test_criterion.text, results)
    print(f'Criterion: {recommendation.criterion_text[:150]}')
    print(f'Label: {recommendation.label}')
    print(f'Confidence: {recommendation.confidence:.2f}')
    print(f'Rationale: {recommendation.rationale}')
    print(f'Evidence trials: {recommendation.evidence_trials}')
    if recommendation.suggested_wording != test_criterion.text:
        print(f'Suggested wording: {recommendation.suggested_wording}')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — full pipeline run
print('Running full CI RAG pipeline...')
results_df = run_ci_rag(
    ie_criteria,
    competing_trials_md,
    persist_dir=str(PERSIST_DIR),
)
print(f'\nResults: {len(results_df)} criteria evaluated')
display(results_df[['criterion_type', 'criterion_text', 'label', 'confidence', 'rationale']].head(10))